# MNIST Digit Classifier — CNN Training & Deployment Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgorume/Mnist_Assignment4.ipynb/blob/main/mnist_full_pipeline.ipynb)

**AI Prompt used to generate this pipeline:**
> *"Create a PyTorch script to fine-tune a pre-trained convolutional neural network for the MNIST dataset. Fine tune for a couple of epochs and save the weights as a .pth file. Deploy the trained CNN model on a Hugging Face Space with Gradio. The Gradio interface should include some examples of input images and should output the estimated probability for each predicted digit. Add a virtual pad to the interface where you can draw a number 0-9 to be classified by the predictive model."*

---
## PART 1 — Train the CNN on MNIST & Save Weights

In [ ]:
# AI Prompt: "Install all required Python libraries for training a PyTorch CNN on MNIST"
!pip install -q torch torchvision matplotlib numpy
print('All dependencies installed!')

In [ ]:
# AI Prompt: "Import all necessary libraries for training a CNN with PyTorch on the MNIST dataset"
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# AI Prompt: "Set the training hyperparameters for fine-tuning a CNN on MNIST for a few epochs"
BATCH_SIZE    = 64
NUM_EPOCHS    = 5
LEARNING_RATE = 0.001
MODEL_PATH    = 'mnist_cnn_weights.pth'

print(f'Batch size:    {BATCH_SIZE}')
print(f'Epochs:        {NUM_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Model will be saved to: {MODEL_PATH}')

In [ ]:
# AI Prompt: "Download and load the MNIST dataset using torchvision with normalization preprocessing"
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Training samples: {len(train_dataset):,}')
print(f'Test samples:     {len(test_dataset):,}')

In [ ]:
# AI Prompt: "Visualise a batch of MNIST training images in a grid using matplotlib"
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {labels[i].item()}')
    ax.axis('off')
plt.suptitle('Sample MNIST Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# AI Prompt: "Define a SimpleCNN convolutional neural network in PyTorch with two conv layers and two fully connected layers for classifying MNIST digits"
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# AI Prompt: "Write a PyTorch training loop with CrossEntropyLoss and Adam optimizer that prints train loss, train accuracy and test accuracy after each epoch"
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc  = 100.0 * correct / total

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
    test_acc = 100.0 * correct / total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    print(f'Epoch [{epoch}/{NUM_EPOCHS}] '
          f'Loss: {train_loss:.4f} | '
          f'Train Acc: {train_acc:.2f}% | '
          f'Test Acc: {test_acc:.2f}%')

print('\nTraining complete!')

In [ ]:
# AI Prompt: "Plot the training loss and accuracy curves side by side using matplotlib"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history['train_loss'], 'b-o')
ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
ax2.plot(epochs, history['test_acc'],  'r-o', label='Test Acc')
ax2.set_title('Accuracy', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('CNN Training on MNIST', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Final Test Accuracy: {history["test_acc"][-1]:.2f}%')

In [ ]:
# AI Prompt: "Save the trained PyTorch model weights to a .pth file using torch.save and download it from Colab"
torch.save(model.state_dict(), MODEL_PATH)
print(f'Model weights saved to: {MODEL_PATH}')

import os
size_mb = os.path.getsize(MODEL_PATH) / 1e6
print(f'File size: {size_mb:.2f} MB')

# Download the .pth file to your computer — upload this to Hugging Face Spaces
try:
    from google.colab import files
    files.download(MODEL_PATH)
    print('Download started — upload this .pth file to your Hugging Face Space!')
except ImportError:
    print('Not in Colab — file saved locally.')

---
## PART 2 — requirements.txt

In [ ]:
# AI Prompt: "Create a requirements.txt file listing the Python dependencies needed to run a PyTorch + Gradio app on Hugging Face Spaces"
requirements = """torch>=2.5.0
torchvision>=0.20.0
Pillow>=9.0.0
numpy>=1.24.0
gradio>=4.0.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('requirements.txt created:')
print(requirements)

---
## PART 3 — app.py (Gradio Deployment Code)

In [ ]:
# AI Prompt: "Write a Gradio app in Python that loads a trained PyTorch CNN model, accepts a drawn digit from a sketchpad or an uploaded image, and outputs the predicted digit and a probability bar chart for all 10 classes (0-9). Include example images for digits 0-9."
app_code = '''
import torch
import torch.nn as nn
from torchvision import transforms
import gradio as gr
import numpy as np
from PIL import Image, ImageOps, ImageDraw, ImageFont
import os
import traceback

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pytorch = SimpleCNN().to(device)
model_save_path = "mnist_cnn_weights.pth"

model_loaded = False
if os.path.exists(model_save_path):
    model_pytorch.load_state_dict(torch.load(model_save_path, map_location=device))
    model_pytorch.eval()
    model_loaded = True
    print(f"Model loaded from {model_save_path}")
else:
    print(f"WARNING: {model_save_path} not found!")

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def make_digit_image(digit, size=280):
    img = Image.new("RGB", (size, size), color=(0, 0, 0))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", size=200)
    except:
        font = ImageFont.load_default()
    text = str(digit)
    bbox = draw.textbbox((0, 0), text, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (size - w) // 2 - bbox[0]
    y = (size - h) // 2 - bbox[1]
    draw.text((x, y), text, fill=(255, 255, 255), font=font)
    return img

os.makedirs("examples", exist_ok=True)
example_paths = []
for d in range(10):
    path = f"examples/digit_{d}.png"
    if not os.path.exists(path):
        make_digit_image(d).save(path)
    example_paths.append(path)

def predict_digit(sketch_input):
    try:
        if not model_loaded:
            return "ERROR: Model weights not found!", {str(i): 0.0 for i in range(10)}
        if sketch_input is None:
            return "Please draw a digit first.", {str(i): 0.0 for i in range(10)}
        if isinstance(sketch_input, dict):
            img_array = sketch_input.get("composite")
            if img_array is None:
                layers = sketch_input.get("layers", [])
                img_array = layers[0] if layers else None
            if img_array is None:
                return "Could not read canvas.", {str(i): 0.0 for i in range(10)}
        else:
            img_array = sketch_input
        if img_array.ndim == 3 and img_array.shape[2] == 4:
            pil_img = Image.fromarray(img_array.astype("uint8"), mode="RGBA").convert("L")
        elif img_array.ndim == 3 and img_array.shape[2] == 3:
            pil_img = Image.fromarray(img_array.astype("uint8"), mode="RGB").convert("L")
        else:
            pil_img = Image.fromarray(img_array.astype("uint8"), mode="L")
        arr = np.array(pil_img)
        if arr.max() == 0:
            return "Canvas appears empty. Please draw a digit.", {str(i): 0.0 for i in range(10)}
        if arr.mean() > 127:
            pil_img = ImageOps.invert(pil_img)
        tensor = transform(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model_pytorch(tensor)
            probs = torch.nn.functional.softmax(output, dim=1)[0]
            predicted = torch.argmax(probs).item()
            confidence = float(probs[predicted]) * 100
        confidences = {str(i): float(probs[i]) for i in range(10)}
        return f"Predicted Digit: {predicted}  (confidence: {confidence:.1f}%)", confidences
    except Exception as e:
        return f"Error: {str(e)}", {str(i): 0.0 for i in range(10)}

def predict_from_image(image_input):
    if image_input is None:
        return "Please upload an image.", {str(i): 0.0 for i in range(10)}
    if isinstance(image_input, np.ndarray):
        pil_img = Image.fromarray(image_input.astype("uint8")).convert("L")
    else:
        pil_img = image_input.convert("L")
    arr = np.array(pil_img)
    if arr.mean() > 127:
        pil_img = ImageOps.invert(pil_img)
    tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model_pytorch(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)[0]
        predicted = torch.argmax(probs).item()
        confidence = float(probs[predicted]) * 100
    confidences = {str(i): float(probs[i]) for i in range(10)}
    return f"Predicted Digit: {predicted}  (confidence: {confidence:.1f}%)", confidences

with gr.Blocks(title="MNIST Digit Classifier") as demo:
    gr.Markdown("# MNIST Digit Classifier")
    gr.Markdown("Choose an input method below, then click Submit to classify.")
    with gr.Tabs():
        with gr.TabItem("Draw a digit"):
            with gr.Row():
                with gr.Column():
                    sketch = gr.Sketchpad(label="Draw a digit here (0-9)", type="numpy", canvas_size=(280, 280))
                    with gr.Row():
                        btn_submit = gr.Button("Submit", variant="primary")
                        btn_clear  = gr.Button("Clear")
                with gr.Column():
                    text_out1  = gr.Textbox(label="Prediction")
                    label_out1 = gr.Label(num_top_classes=10, label="Probability per digit (0-9)")
            btn_submit.click(fn=predict_digit, inputs=sketch, outputs=[text_out1, label_out1])
            btn_clear.click(fn=lambda: (None, "", {}), outputs=[sketch, text_out1, label_out1])
        with gr.TabItem("Upload or use example"):
            with gr.Row():
                with gr.Column():
                    image_input = gr.Image(label="Upload an image of a handwritten digit", type="numpy")
                    btn_submit2 = gr.Button("Submit", variant="primary")
                with gr.Column():
                    text_out2  = gr.Textbox(label="Prediction")
                    label_out2 = gr.Label(num_top_classes=10, label="Probability per digit (0-9)")
            btn_submit2.click(fn=predict_from_image, inputs=image_input, outputs=[text_out2, label_out2])
            gr.Markdown("### Example digits — click any image to load and classify it")
            gr.Examples(
                examples=[[p] for p in example_paths],
                inputs=image_input,
                outputs=[text_out2, label_out2],
                fn=predict_from_image,
                cache_examples=True,
            )
demo.launch()
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('app.py written to disk successfully!')
print('--- app.py contents ---')
with open('app.py') as f:
    print(f.read())

In [ ]:
# AI Prompt: "Verify that all three deployment files exist on disk and show their file sizes"
import os
print('Files ready for Hugging Face Spaces deployment:')
print('-' * 50)
for fname in ['mnist_cnn_weights.pth', 'requirements.txt', 'app.py']:
    exists  = os.path.exists(fname)
    size_mb = os.path.getsize(fname) / 1e6 if exists else 0
    status  = f'YES  ({size_mb:.2f} MB)' if exists else 'MISSING'
    print(f'  {fname:30s}  {status}')
print('-' * 50)
print('Upload all three files to your Hugging Face Space to deploy the app.')

---
## Summary

This notebook produced **3 files** to upload to Hugging Face Spaces:

| File | Description |
|------|-------------|
| `mnist_cnn_weights.pth` | Trained CNN weights saved with `torch.save(model.state_dict())` |
| `requirements.txt` | Python dependencies for the Hugging Face Space |
| `app.py` | Gradio app that loads weights with `model.load_state_dict()` and serves predictions |

**Hugging Face Space:** https://huggingface.co/spaces/YOUR_USERNAME/mnist-digit-classifier